In [1]:
from datascience import *

In [2]:
import random
import numpy as np 

In [22]:
# t: inventory
# d: demand of customer (unknown in the first state)
# s: sales in one day
# r: revenue after each day
# p: price
# state (remaining inventory, time before expiration)
# action: increase/decrease/keep the price
# reward: reward: increase revenue, penalty: decrease/unchanged
# transaction prob: increase revenue/decrease/unchanged

In [3]:
def get_dataset(inventory):
    customer_demand = np.empty(shape=0,dtype=int) #create customer demand array
    sales = np.empty(shape=0,dtype=int) #create sales array
    price = np.empty(shape=0,dtype=float) #create price array
    inventory_arr = np.empty(shape=0,dtype=int) #create inventory array
    purchase_prob_arr = np.empty(shape=0,dtype=float)
    date = 30
    date_arr = np.empty(shape=0,dtype=int)
    for day in range(0,31):
        day_demand = random.randint(0,60) #simulate daily demand in a day 
        customer_demand =np.append(customer_demand,day_demand)

        day_price = random.randint(0,3000) /100 #simulate daily price in a day 
        price = np.append(price, day_price)

        #find the probability of customers making the sales and times the day demand to get actual day sale
        purchase_probability = round((-1/900) * (day_price)**2 +1,2) #find the probability of customers purchasing (decaying)
        purchase_prob_arr = np.append(purchase_prob_arr,purchase_probability) #add to the array
        day_sale = round(purchase_probability*day_demand) #calculate the sales
        if day_sale<0: #if there are no customers visit the website/sale yield negative, no sales were made
            day_sale = 0
            sales = np.append(sales, day_sale)
            remaining_inventory = inventory - day_sale
        else:
            remaining_inventory = inventory - day_sale
            if remaining_inventory < 0: #negative inventory, set the day sale to the remaining inventory and recalculate the remaining inventory, which should yield 0
                day_sale = inventory
                remaining_inventory = inventory - day_sale  #recalculate the remaining inventory
            sales = np.append(sales, day_sale) #append the sale to the list

        inventory_arr = np.append(inventory_arr,remaining_inventory) #append the inventory to the array
        inventory = remaining_inventory #update the current inventory
        date_arr = np.append(date_arr,date)
        date-=1
        if inventory == 0:
            break

    revenue = price * sales #calculate the revenue
    table = Table().with_columns (
        'Date', date_arr,
        'Price', price,
        'Customer Demand', customer_demand,
        'Purchase Probability', purchase_prob_arr,
        'Sales', sales,
        'Revenue',revenue,
        'Remaining Inventory', inventory_arr
    )
    #adding the profit column
    revenue_diff_arr = np.array([revenue[0]])
    range_len = len(table.column('Revenue'))-1
    for r in range (0,range_len):
        revenue_diff = table.column('Revenue')[r+1]-table.column('Revenue')[r]
        # print(revenue_diff)
        revenue_diff_arr = np.append(revenue_diff_arr,revenue_diff)
    #print(revenue_diff_arr)
    new_table = table.with_column(
        'Profit',revenue_diff_arr
    )
    return new_table

In [4]:
simulated_dataset = get_dataset(1000)
simulated_dataset.show()

Date,Price,Customer Demand,Purchase Probability,Sales,Revenue,Remaining Inventory,Profit
30,24.93,19,0.31,6,149.58,994,149.58
29,6.96,23,0.95,22,153.12,972,3.54
28,1.64,11,1,11,18.04,961,-135.08
27,11.86,30,0.84,25,296.5,936,278.46
26,29.82,29,0.01,0,0,936,-296.5
25,3.55,15,0.99,15,53.25,921,53.25
24,23.54,10,0.38,4,94.16,917,40.91
23,15.97,58,0.72,42,670.74,875,576.58
22,8.53,25,0.92,23,196.19,852,-474.55
21,19.25,44,0.59,26,500.5,826,304.31


In [5]:
def compare(old,new):
    if old<new:
        return "Increasing"
    elif old>new:
        return "Decreasing"
    else:
        return "Unchanged" 

In [6]:
def get_price_table(data):
    price = data.column('Price')
    new_price = np.append(price[1:len(price)+1],0)
    price_table = Table().with_columns(
    'Old Price',price,
    'New Price',new_price
    )
    price_table = price_table.with_column(
    "Comparison", price_table.apply(compare,'Old Price','New Price')
    )
    return price_table

In [7]:
price_table = get_price_table(simulated_dataset)

In [8]:
price_table.show()

Old Price,New Price,Comparison
24.93,6.96,Decreasing
6.96,1.64,Decreasing
1.64,11.86,Increasing
11.86,29.82,Increasing
29.82,3.55,Decreasing
3.55,23.54,Increasing
23.54,15.97,Decreasing
15.97,8.53,Decreasing
8.53,19.25,Increasing
19.25,11.78,Decreasing


Conduct __A/B testing__ to see the chance that the future dataset will likely behave the same as this dataset

* Null Hypothesis: The future data will behave the same

* Alternative: The future data won't behave the same

In [86]:
def difference_of_means(tbl, label_column):
    means_table = tbl.group(label_column, np.average)
    means_array = means_table.column(1)
    return means_array.item(1) - means_array.item(0)

In [92]:
observed_difference = difference_of_means(price_table,'Comparison')
observed_difference

-7.448541666666667

In [93]:
def get_p_value(repetition):
    price_table = get_price_table(simulated_dataset)
    observed_difference = difference_of_means(price_table,'Comparison')
    print(observed_difference)
    differences = make_array()

    # for many iterations
    for i in np.arange(repetition):

        # create a new table whose first column is shuffled category labels and second column is old price in original order
        shuffled_comparison = price_table.sample(with_replacement=False)
        combined = price_table.with_column('Shuffled Comparison', shuffled_comparison.column(2))
        combined = combined.select('Shuffled Comparison', 'Old Price')

        # calculate the value of the test statistic for the shuffled versions
        simulated_difference = difference_of_means(combined, 'Shuffled Comparison')

        # append the outcome to array
        differences = np.append(differences, simulated_difference)
    
    #calculate the p-value
    p_value = np.count_nonzero(differences<=observed_difference)/repetition
    print(differences)
    return p_value

In [94]:
get_p_value(1000)

-7.448541666666667
[  3.46666667e-01  -3.98558333e+00  -2.42008333e+00  -2.25087500e+00
  -3.75437500e+00  -9.17875000e-01   7.60000000e-01   4.24620833e+00
   5.57404167e+00   6.77916667e+00   2.07879167e+00  -6.73941667e+00
   2.19375000e+00  -4.56941667e+00  -3.37333333e+00   1.65254167e+00
  -1.67608333e+00  -1.61279167e+00  -1.08792083e+01  -8.33916667e-01
  -2.65583333e-01  -1.35187500e+00   1.23533333e+00  -1.76779167e+00
  -3.19637500e+00   2.89900000e+00  -1.70191667e+00   1.75070833e+00
   3.26000000e-01  -1.78200000e+00   6.07133333e+00   6.38133333e+00
   3.00104167e+00   4.02275000e+00  -4.76125000e-01  -2.55054167e+00
   1.18495833e+00   5.83041667e-01   2.53666667e-01   7.75500000e-01
   1.24437500e+00   3.45570833e+00  -1.01087500e+00   5.00054167e+00
   3.24129167e+00   4.82229167e+00   4.64275000e+00   1.30379167e+00
  -3.81766667e+00  -5.05637500e+00  -2.58800000e+00  -1.45391667e+00
  -2.36066667e+00   1.71066667e+00  -3.21962500e+00  -3.80087500e+00
  -1.23691667e+

0.004

* If p value is 0.05, it is statistically significant to reject the null value

* If p value is 0.01, it is highly statistically significant to reject the null value

In [95]:
def transition_probability(table):
    increase_to_increase = 0
    increase_to_decrease = 0
    increase_to_unchange = 0
    decrease_to_increase = 0
    decrease_to_decrease = 0
    decrease_to_unchange = 0
    unchange_to_increase = 0
    unchange_to_decrease = 0
    unchange_to_unchange = 0

    #check the following action of the price after increasing/decreasing/unchange
    for i in range(len(table['Comparison'])-1):
        if table['Comparison'][i] == "Decreasing":
            if table['Comparison'][i+1]=="Decreasing":
                decrease_to_decrease +=1
            elif table['Comparison'][i+1]=="Increasing":
                decrease_to_increase+=1
            else:
                decrease_to_unchange+=1
        elif table['Comparison'][i] == "Increasing":
            if table['Comparison'][i+1]=="Increasing":
                increase_to_increase +=1
            elif table['Comparison'][i+1]=="Decreasing":
                increase_to_decrease+=1
            else:
                increase_to_unchange+=1
        else:
            if table['Comparison'][i+1]=="Unchanged":
                unchange_to_unchange +=1
            elif table['Comparison'][i+1]=="Increasing":
                unchange_to_increase+=1
            else:
                unchange_to_decrease+=1
    increase = increase_to_increase+increase_to_decrease+increase_to_unchange
    decrease = decrease_to_decrease+decrease_to_increase+decrease_to_unchange 
    unchange = unchange_to_decrease+unchange_to_increase+unchange_to_unchange

    #find the probability
    if increase != 0:
        increase_to_increase_prob = round(increase_to_increase/increase,2)
        increase_to_decrease_prob = round(increase_to_decrease/increase,2)
        increase_to_unchange_prob = round(increase_to_unchange/increase,2)
    else:   
        increase_to_increase_prob = 0
        increase_to_decrease_prob = 0
        increase_to_unchange_prob = 0 

    if decrease!=0:
        decrease_to_decrease_prob = round(decrease_to_decrease/decrease,2)
        decrease_to_increase_prob = round(decrease_to_increase/decrease,2)
        decrease_to_unchange_prob = round(decrease_to_unchange/decrease,2)
    else:
        decrease_to_decrease_prob = 0
        decrease_to_increase_prob = 0
        decrease_to_unchange_prob = 0
    if unchange!=0:
        unchange_to_decrease_prob = round(unchange_to_decrease/unchange,2)
        unchange_to_increase_prob = round(unchange_to_increase/unchange,2)
        unchange_to_unchange_prob = round(unchange_to_unchange/unchange,2)
    else:
        unchange_to_decrease_prob = 0
        unchange_to_increase_prob = 0
        unchange_to_unchange_prob = 0
    return increase_to_increase_prob,increase_to_decrease_prob,increase_to_unchange_prob,decrease_to_decrease_prob,decrease_to_increase_prob,decrease_to_unchange_prob,unchange_to_decrease_prob,unchange_to_increase_prob,unchange_to_unchange_prob

In [99]:
transition_probability(price_table)

(0.44, 0.56, 0.0, 0.36, 0.64, 0.0, 0, 0, 0)

In [9]:
def get_state_index(index_state,grid_size):
    return index_state[0] * grid_size[1] + index_state[1] #get the index to find the corresponding profit and value

In [10]:
def get_price(action,price):
    if action == 0:
        price +=1
    elif action == 1:
        price
    elif action == 2:
        price -= 1
    return price

In [11]:
def get_next_state1(purchase_prob, customers_demand,inventory,day):
    #x is price, y is reman
    remaining_inv = inventory - (purchase_prob*customers_demand) #find the remaining inventory for each action
    day-=1 #decrement day bt 1, we have 30 days remaining
    return (day,remaining_inv) #next state

In [12]:
def get_reward1(price, purchase_prob, customers_demand):
    reward = price*purchase_prob*customers_demand
    return reward

In [13]:
price_probability = dict()

In [14]:
def get_purchase_probability(price,data):
    global price_probability
    if price in price_probability.keys():
        purchase_probability = price_probability[price] #look into the dictionary with price-purchase probability pair to see if we have already have them
    else:
        data_match = data.where('Price',are.below_or_equal_to(price)).column('Price')
        if len(data_match) == 0:
            closest_match = min(data.where('Price',are.above_or_equal_to(price)).column('Price')) #sometimes action -1 dollar make the price to be the smallest, this is when we take the closest larget price
        else:
            closest_match = max(data_match) #find the closest smaller price to the new price
    #get purchase probability
        purchase_probability = data.where('Price',closest_match).column('Purchase Probability')[0]
        price_probability[price]=purchase_probability
    return purchase_probability

In [11]:
get_purchase_probability(24.3,simulated_dataset)

0.44

In [15]:
def get_value(price, day_remain, inv_remain,purchase_porb,customers_demand,data, depth):
    
    if day_remain < 0 or inv_remain < 0 or depth >10:
        return get_reward1(price,purchase_porb,customers_demand) #base case
    else:
        reward = get_reward1(price,purchase_porb,customers_demand) #get the reward

        #find the purchase probability at increased/unchaged/decreased price
        purchase_prob_at_increased_price = get_purchase_probability(price+1,data)
        purchase_prob_at_unchanged_price = get_purchase_probability(price,data)
        purchase_prob_at_decreased_price = get_purchase_probability(price-1,data)
        
        value= reward + 0.9 / 3 * (
            get_value(price + 1, day_remain - 1, inv_remain - customers_demand * purchase_prob_at_increased_price,purchase_prob_at_increased_price,30,data,depth+1) 
            + get_value(price, day_remain - 1, inv_remain - customers_demand * purchase_prob_at_unchanged_price,purchase_prob_at_unchanged_price,30,data,depth+1) 
            + get_value(price - 1, day_remain - 1, inv_remain - customers_demand * purchase_prob_at_decreased_price,purchase_prob_at_decreased_price,30,data,depth+1))

        return value

In [16]:
def get_value_2(price, day_remain, inv_remain,purchase_porb,customers_demand,data, depth):
    
    if day_remain < 0 or inv_remain < 0 or depth >10:
        return get_reward1(price,purchase_porb,customers_demand) #base case
    else:
        reward = get_reward1(price,purchase_porb,customers_demand) #get the reward

        #find the purchase probability at increased/unchaged/decreased price
        purchase_prob_at_increased_price = get_purchase_probability(price+1,data)
        purchase_prob_at_unchanged_price = get_purchase_probability(price,data)
        purchase_prob_at_decreased_price = get_purchase_probability(price-1,data)
        
        value= reward + max(
            get_value(price + 1, day_remain - 1, inv_remain - customers_demand * purchase_prob_at_increased_price,purchase_prob_at_increased_price,30,data,depth+1) 
            ,get_value(price, day_remain - 1, inv_remain - customers_demand * purchase_prob_at_unchanged_price,purchase_prob_at_unchanged_price,30,data,depth+1) 
            ,get_value(price - 1, day_remain - 1, inv_remain - customers_demand * purchase_prob_at_decreased_price,purchase_prob_at_decreased_price,30,data,depth+1))

        return value

In [17]:
def value_iteration(data):
    date_range = data.column('Date') #x
    inventory_range = data.column('Remaining Inventory') #y
    chosen_price = data.column('Price')[0] #set the initial price to the first price in the dataset
    grid_size = (len(date_range),len(inventory_range))
    num_actions = 3
    Value = np.zeros(grid_size[0]) #create a place holder for calculated value
   
    for day in range(grid_size[0]):
        #for inv in range(grid_size[1]):
            value_state = (date_range[day],inventory_range[day]) #do i need this?
            #state_index = get_state_index(index_state,grid_size) #get the index of the location to update this location later
            v = Value[day] #get the number out (0 most of the time until loop through the second time)
            print(f'This is v: {v}')
            max_value = float('-inf')
                   
            for action in range(num_actions): #
                    print(f'Action: {action}')
                    print('Price last stage', chosen_price)
                    price = chosen_price #update the price to the price yield the most value
                    price = get_price(action,price) #update the price
                    
                    #get purchase probability
                    purchase_probability = get_purchase_probability(price,data)
                    
                    #get next state
                    next_state = get_next_state1(purchase_probability,30,value_state[1],value_state[0]) #if took that action, the new pair (x,y) is the remaining day and the remaining inventory
                    print(f'Next state {next_state}')


                    value = get_value(price,next_state[0],next_state[1],purchase_probability,30,data,0) #calcute with the reward of the current pair and the value of the next pair
                    print(f'This is value {value}')
                    if value > max_value: #if find bigger value, update
                        max_value = value
                        chosen_price = price #keep the price the yield the most value
                    
                    Value[day] = max_value #update the value in the current pair whenever we found a bigger value
    print(Value)

In [ ]:
value_iteration(simulated_dataset)

In [18]:
action_dict = {
    0: "Increase $1",
    1:"Keep price",
    2:"Decrease $1"}

In [52]:
action_dict

{0: 'Increase $1', 1: 'Keep price', 2: 'Decrease $1'}

In [21]:
def policy_iteration(data):
    date_range = data.column('Date') #x
    inventory_range = data.column('Remaining Inventory') #y
    chosen_price = data.column('Price')[0] #set the initial price to the first price in the dataset
    grid_size = (len(date_range),len(inventory_range))
    num_actions = 3
    optimal_policy = np.empty(shape=(0),dtype=tuple)
   
    for day in range(grid_size[0]):
        value_state = (date_range[day],inventory_range[day])
            
        max_value = float('-inf')
        price = chosen_price #update the price to the price yield the most value
        #print(chosen_price)
        for action in range(num_actions): 
            print(f'Action: {action}')
            print('Price last stage', price)
            new_price = get_price(action,price) #update the price
            print('Price this stage:', new_price)
            #get purchase probability
            purchase_probability = get_purchase_probability(new_price,data) #get purchase probability at the price after action taken
                    
            #get next state
            next_state = get_next_state1(purchase_probability,30,value_state[1],value_state[0]) #if took that action, the new pair (x,y) is the remaining day and the remaining inventory
            #print(f'Next state {next_state}')

            #value = get_value(new_price,next_state[0],next_state[1],purchase_probability,30,data,0) #calcute with the reward of the current pair and the value of the next pair
            value = get_value_2(new_price,next_state[0],next_state[1],purchase_probability,30,data,0) #calcute with the reward of the current pair and the value of the next pair
            print(f'This is value {value}')
            if value > max_value: #if find bigger value, update
                max_value = value
                chosen_price = new_price #keep the price the yield the most value
                best_action = action
                revenue = chosen_price * purchase_probability * 30 #calculate revenue if the action is taken in the next stage
        #print(chosen_price)
        optimal_policy = np.append(optimal_policy,[value_state[0],action_dict[best_action],chosen_price,revenue]) #add to the optimal policy
    print(optimal_policy)

In [17]:
simulated_dataset = get_dataset(1000)

In [22]:
policy_iteration(simulated_dataset)

Action: 0
Price last stage 24.93
Price this stage: 25.93
This is value 1842.3035533156958
Action: 1
Price last stage 24.93
Price this stage: 24.93
This is value 2094.584445301933
Action: 2
Price last stage 24.93
Price this stage: 23.93
This is value 2347.6836400149205
Action: 0
Price last stage 23.93
Price this stage: 24.93
This is value 2094.584445301933
Action: 1
Price last stage 23.93
Price this stage: 23.93
This is value 2347.6836400149205
Action: 2
Price last stage 23.93
Price this stage: 22.93
This is value 2528.411072512648
Action: 0
Price last stage 22.93
Price this stage: 23.93
This is value 2347.6836400149205
Action: 1
Price last stage 22.93
Price this stage: 22.93
This is value 2528.411072512648
Action: 2
Price last stage 22.93
Price this stage: 21.93
This is value 2675.2362521899813
Action: 0
Price last stage 21.93
Price this stage: 22.93
This is value 2528.411072512648
Action: 1
Price last stage 21.93
Price this stage: 21.93
This is value 2675.2362521899813
Action: 2
Price

In [ ]:
def customer_demand(price):
    return round((-1/900) * price**2 +1,2)


def value(price, day_remain, inv_remain,data):
    if day_remain < 0 or inv_remain < 0:
        return 0
    else:
        print(price,day_remain, inv_remain)
        return price * 30 * customer_demand(price) + 0.3 / 3 * (value(price + 1, day_remain - 1, inv_remain - 30 * get_purchase_probability(price+1,data),data) 
                                                                + value(price, day_remain - 1, inv_remain - 30 * get_purchase_probability(price,data),data) 
                                                                + value(price - 1, day_remain - 1, inv_remain - 30 * get_purchase_probability(price-1,data),data))

In [ ]:
def value1(price, day_remain, inv_remain,data):
    global price_value
    if price in price_value.keys():
        return price_value[price]
    elif day_remain < 0 or inv_remain < 0:
        return 0
    else:
        print(price,day_remain, inv_remain)
        value = price * 30 * get_purchase_probability(price,data) + 0.3 / 3 * (value1(price + 1, day_remain - 1, inv_remain - 30 * get_purchase_probability(price+1,data),data) 
                                                                + value1(price, day_remain - 1, inv_remain - 30 * get_purchase_probability(price,data),data) 
                                                                + value1(price - 1, day_remain - 1, inv_remain - 30 * get_purchase_probability(price-1,data),data))
        price_value[price] = value
        return value

In [ ]:
value1(15, 30, 1000,simulated_dataset)

483.27834133453723